# ![icon](./images/uva-icon-57x57.png) WEEK 10 Gitops in Snowflake

| **Working Efficiently With Software** |
| :--: |
| **Data Engineering** |
| **School of Data Science** |
| **University of Virginia** |

# ![icon](./images/uva-icon-57x57.png) Announcements & Agenda

---

## **Looking for volunteers to talk about the use of Claude in the class**
**We need at least 2 students.  3 better, but 2 is good**


---

## **Question about docker lab**
**Check your docker file and makefile into a branch just like previou labs.  Put your link to the dockerhub image in the PR description**

---


## T-Shirt sizing the remaining labs.
- ~~Setting up snowflake connector.  **M/L**~~
- ~~Dockerizing our pipeline.~~  **XL**
- Git repo in snowflake.  **M** **<===== AWS ⬆️ == Snowflake ⬇️ ===**
- DBT-expectations **M**

## We'll probably do this lab "in class" on the 4th.
- Snowflake CORTEX.  **S**


## Participation Reminder
**Can you see me?  I realize my video has been showing up blank in some meetings**
* Class attendance
* Cameras on please

# ![icon](./images/uva-icon-57x57.png) GitOps Infrastructure & Native Snowflake Integration

## Links
* [snowflake](https://app.snowflake.com/adilksi/fqa59308/#/workspaces/ws/USER%24/PUBLIC/DS5111_Workspace/05_git_integration.sql)
* [github](https://github.com/Niarfe/2605_DS5111_dpywq/tree/LAB09_gitops_snowflake/transform)

# ![icon](./images/uva-icon-57x57.png) WHY: Moving Beyond "Copy-Paste Data Warehouse Management"

Historically, database scripts were run manually: developers edited SQL locally, pasted it into a web UI worksheet, and hit run. This leads to:

* **"Works on my machine" queries** with no version history or audit trail.
* **Schema drift** between dev, staging, and production.
* **Accidental production overrides** when running scripts in the wrong database/schema context.

# **GitOps**
| **Applies software engineering best practices**| 
| :-: |
| **Version Control**| 
| **Code Review**| 
| **CI/CD**| 
| **Environmental Isolation**| 
| **directly to database transformations and infrastructure.**| 

# ![icon](./images/uva-icon-57x57.png) WHAT: Native Snowflake Git Integration

Snowflake allows us to treat a Git repository as a **First-Class Stage Object** (`GIT REPOSITORY`).

* Instead of running local files or uploading `.sql` files manually, Snowflake connects directly to GitHub.
* Branches, tags, and commits are exposed as virtual directory paths inside Snowflake (e.g., `@<repo>/branches/<branch_name>/scripts/`).

**This functionality has to activated.  As an Admin I activated your roles**

**NB:** The `API INTEGRATION` and the link to your role `ds5111_student_role`
```sql
USE ROLE ACCOUNTADMIN;

-- Create a reusable API integration dedicated to handling Git network requests
CREATE OR REPLACE API INTEGRATION github_public_integration
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/')
  ENABLED = true;

-- Hand ownership over to SYSADMIN so your engineering roles can use it
GRANT USAGE ON INTEGRATION github_public_integration TO ROLE SYSADMIN;
GRANT USAGE ON INTEGRATION github_public_integration TO ROLE ds5111_student_role;

```

# ![icon](./images/uva-icon-57x57.png) HOW: The GitOps Workflow for Data Engineers

1. **Develop:** Write SQL/dbt logic locally on a **[FEATURE BRANCH](https://github.com/Niarfe/2605_DS5111_dpywq/tree/LAB09_gitops_snowflake/transform)**
2. **Push:** Push changes to GitHub.
3. **Fetch:** Instruct Snowflake to fetch the latest commits (`ALTER GIT REPOSITORY ... FETCH`).
4. **Test:** Execute scripts from the branch stage into an isolated dev schema (`DEV_<UVAID>`).
5. **Merge & Deploy:** Merge PR to main (`LAB09_gitops_snowflake`) and run against the primary schema (`<UVAID>`).

## Clone the github repo into a database schema object

**NB:** Use of the role `ds5111_student_role` has been granted github integration priviliges
```sql
USE ROLE ds5111_student_role;
USE DATABASE ds5111_db;
USE SCHEMA txt1sr; -- Substitute with active user schema workspace

-- Create the database-level Git Repository tracking object
CREATE OR REPLACE GIT REPOSITORY student_pipeline_repo
  API_INTEGRATION = github_public_integration
  ORIGIN = 'https://github.com/Niarfe/2605_DS5111_dpy8wq'; 

```

## Fetch updates and inspecting your file system
```sql
-- 1. Explicitly pull the latest commit history down from GitHub across the wire
ALTER GIT REPOSITORY student_pipeline_repo FETCH;

-- 2. Check what branches exist inside the tracked database object
SHOW GIT BRANCHES IN student_pipeline_repo;

-- 3. Treat the main branch like a stage and list the files sitting in your repo
LS @student_pipeline_repo/branches/main/;

-- 4. Drill down specifically into your pipeline directory structure
LS @student_pipeline_repo/branches/main/labs/;

```

## Execute from github repo
```sql
-- Execute a raw SQL setup file straight out of the GitHub main branch
EXECUTE IMMEDIATE FROM @student_pipeline_repo/branches/main/labs/lab7_setup_fixtures.sql;

```

# ![icon](./images/uva-icon-57x57.png) Infrastructure Setup: Public Repos in Snowflake

* **Why Public Repos?** Snowflake Git integrations for *private* repositories require creating a Snowflake `SECRET` (storing a GitHub Personal Access Token) and an `API_INTEGRATION` object requiring elevated admin privileges.
* **Public Repo Convenience:** By using a public GitHub repository HTTPS URL, Snowflake can clone and fetch remote refs directly without requiring user-level credentials or secret management in the user session.

## I set up the lab to use public repos, however, the accepted practice is to use private repos.  I left instructions for that as an alternative.


# ![icon](./images/uva-icon-57x57.png) Lab branching

**This weeks lab asks you to branch from your feature branch, then back INTO your feature branch**

This is just to show you how to coordinate git/github with your changes.  The reason snowlake integrated git in the first place.

**SQL IS CODE.  SO TREAT IT LIKE CODE.  VERSION CONTROL.  REVIEW.**


# ![icon](./images/uva-icon-57x57.png) Core Architectural Rules for Lab 9

To ensure environmental safety and eliminate bugs during the lab, we enforce three strict design patterns:

### Pattern A: Environmental Isolation (Dual-Schema Pattern)

* **`<UVAID>` Schema:** Your primary, production-like schema. Only thoroughly tested code merged into the main lab branch is deployed here.
* **`DEV_<UVAID>` Schema:** Your playground schema. Feature branches and experimental SQL scripts **must** be executed here first.

### Pattern B: Fully Qualified Pathing Strategy

Context-based errors (e.g., running a script in the wrong database because `USE SCHEMA` was missing) are the #1 source of database outages.

* **Rule:** Always use fully qualified object references:
`DS5111_DB.<UVAID>.RAW_TRANSCRIPTS`
* **Rule:** Always use fully qualified stage paths:
`@DS5111_DB.<UVAID>.DS5111_GIT_STAGE/branches/feature-my-branch/scripts/deploy.sql`

### Pattern C: Hyphenated Branch Naming

* Standard Git convention often uses slashes for feature branches (e.g., `feature/add-summary`).
* **Snowflake Stage Pitfall:** Snowflake parses forward slashes (`/`) as stage directory separators! A branch named `feature/add-summary` gets parsed as subfolder `/add-summary` inside folder `/feature`.
* **Rule:** Use hyphens instead of slashes for all Git branches in this lab:
`feature-add-char-count` ✅

# ![icon](./images/uva-icon-57x57.png)

### Code Example: Executing SQL Directly from a Git Stage

```sql
-- 1. Sync Snowflake with the latest commits pushed to GitHub
ALTER GIT REPOSITORY DS5111_DB.<UVAID>.DS5111_GIT_STAGE FETCH;

-- 2. Inspect files in your feature branch
LIST @DS5111_DB.<UVAID>.DS5111_GIT_STAGE/branches/feature-add-char-count/;

-- 3. Execute a SQL deployment script directly from your feature branch into your DEV schema
USE SCHEMA DS5111_DB.DEV_<UVAID>;

EXECUTE IMMEDIATE FROM '@DS5111_DB.<UVAID>.DS5111_GIT_STAGE/branches/feature-add-char-count/scripts/01_create_feature_table.sql';

```

---

# ![icon](./images/uva-icon-57x57.png)

## 4. Student "Heads-Up" & Common Lab Gotchas

Keep these frequent pain points in mind when working through Lab 9:

### ⚠️ Gotcha 1: The "Invisible Code" Bug (Forgetting `FETCH`)

Pushing your code to GitHub does **not** automatically push it into Snowflake. Snowflake's Git stage is a snapshot.

* **Symptom:** You edited a file, pushed to GitHub, but `EXECUTE IMMEDIATE FROM` in Snowflake still runs the old code.
* **Fix:** Always run `ALTER GIT REPOSITORY <stage_name> FETCH;` before testing your changes in Snowflake.

### ⚠️ Gotcha 2: Branch Pathing Errors (Slash vs Hyphen)

* **Symptom:** `Stage file does not exist` error when running `EXECUTE IMMEDIATE FROM`.
* **Fix:** Verify your branch name uses hyphens (`feature-foo`), not forward slashes (`feature/foo`).

### ⚠️ Gotcha 3: Schema Leakage / Context Errors

* **Symptom:** Your feature testing modified base tables in your primary schema instead of your dev schema.
* **Fix:** Check your script header! Always explicitly issue `USE SCHEMA DEV_<UVAID>;` or use fully qualified table names (`DEV_<UVAID>.MY_TABLE`).

### ⚠️ Gotcha 4: Local Git vs Remote Git

Remember that Snowflake fetches from **GitHub (the remote origin)**, not your local Mac/PC file system.

* **Fix:** If you haven't run `git push origin feature-branch-name`, Snowflake has no way of seeing your local code changes.

---


## JINJA AND DBT PREVIEW

# ![icon](./images/uva-icon-57x57.png) What is Jinja Templating?

## Giving SQL "Programming Superpowers"

SQL is declarative and powerful, but standard SQL is **static**—it lacks variables, loops, conditionals, and reusable modules.

**Jinja** is a templating language for Python that dbt uses to turn static `.sql` files into dynamic code generators. Before dbt sends any SQL to Snowflake, it evaluates all Jinja expressions and compiles them into clean, standard SQL.

---

### The 3 Core Jinja Delimiters

| Syntax | Type | Purpose | Example |
| :--- | :--- | :--- | :--- |
| **`{{ ... }}`** | **Expressions** | Prints a value, variable, or macro result into the SQL output. | `SELECT {{ 5 + 5 }};` |
| **`{% ... %}`** | **Control Flow** | Handles loops, `if/else` logic, and variable assignments. | `{% if target.name == 'dev' %}` |
| **`{# ... #}`** | **Comments** | Notes ignored by both Jinja compilation and Snowflake. | `{# This won't appear in compiled SQL #}` |

---

# ![icon](./images/uva-icon-57x57.png) Jinja Mechanics — Static vs. Dynamic SQL

### The Problem: Manual Repetition in SQL
Imagine pivoting metrics across multiple categories or channels in standard SQL:

```sql
-- Standard SQL: Manual, repetitive, error-prone
SELECT
    video_id,
    SUM(CASE WHEN metric_type = 'views' THEN metric_value END) AS total_views,
    SUM(CASE WHEN metric_type = 'likes' THEN metric_value END) AS total_likes,
    SUM(CASE WHEN metric_type = 'comments' THEN metric_value END) AS total_comments
FROM raw_video_metrics
GROUP BY 1;
```

---

### The Solution: DRY Code with Jinja Loops

```sql
-- Jinja-enhanced SQL (dbt Source File)
{% set metrics = ['views', 'likes', 'comments'] %}

SELECT
    video_id,
    {% for metric in metrics %}
    SUM(CASE WHEN metric_type = '{{ metric }}' THEN metric_value END) AS total_{{ metric }}{% if not loop.last %},{% endif %}
    {% endfor %}
FROM raw_video_metrics
GROUP BY 1;
```

> **Key Takeaway:** Jinja allows you to keep your code **DRY** (*Don't Repeat Yourself*). Changing or adding a metric only requires updating the list array once!

---

# ![icon](./images/uva-icon-57x57.png) dbt + Jinja — How Models & Lineage Work

dbt uses Jinja to do two critical jobs simultaneously:
1. **Dynamic Name Resolution:** Automatically prefixes schema names (`DEV_<UVAID>` vs `<UVAID>`) depending on your target environment.
2. **DAG Building:** The `{{ ref('model_name') }}` function tells dbt which models depend on each other, automatically building the **Lineage Graph (DAG)** and determining execution order.


```mermaid
graph LR
    A[(RAW_TRANSCRIPTS)] -->|"{{ source() }}"| B[stg_youtube_transcripts.sql]
    B -->|"{{ ref() }}"| C[dim_videos.sql]
    B -->|"{{ ref() }}"| D[fct_transcript_sentiment.sql]
    C -->|"{{ ref() }}"| E[rpt_channel_performance.sql]
    D -->|"{{ ref() }}"| E

    style A fill:#f9f,stroke:#333,stroke-width:2px
    style B fill:#bbf,stroke:#333,stroke-width:2px
    style C fill:#bbf,stroke:#333,stroke-width:2px
    style D fill:#bbf,stroke:#333,stroke-width:2px
    style E fill:#bfb,stroke:#333,stroke-width:2px
```

---

# ![icon](./images/uva-icon-57x57.png) dbt Models in Action — Real-World Example

Below is a standard dbt staging model combining the **3 most common Jinja use cases**: model configuration, environment/source references, and conditional logic.

```sql
-- models/staging/stg_youtube_transcripts.sql

-- 1. CONFIG MACRO: Materialization strategy & target schema context
{{ config(
    materialized='view',
    schema=target.schema
) }}

WITH raw_source AS (
    -- 2. REF / SOURCE MACRO: Decouples hardcoded DB/Schema paths
    SELECT * 
    FROM {{ source('youtube_raw', 'raw_transcripts') }}
)

SELECT
    video_id,
    channel_id,
    TRIM(transcript_text) AS cleaned_text,
    LENGTH(cleaned_text) AS char_count,
    fetched_at
FROM raw_source

-- 3. CONTROL FLOW: Limit data volume in Dev environments to save compute
{% if target.name == 'dev' %}
WHERE fetched_at >= DATEADD('day', -7, CURRENT_DATE())
{% endif %}
```

---

### What dbt Compiles & Sends to Snowflake (In Dev Environment):

```sql
-- Target Compiled SQL executed in Snowflake
CREATE OR REPLACE VIEW DS5111_DB.DEV_UVAID.STG_YOUTUBE_TRANSCRIPTS AS (
    WITH raw_source AS (
        SELECT * 
        FROM DS5111_DB.RAW_DATA.RAW_TRANSCRIPTS
    )
    SELECT
        video_id,
        channel_id,
        TRIM(transcript_text) AS cleaned_text,
        LENGTH(cleaned_text) AS char_count,
        fetched_at
    FROM raw_source
    WHERE fetched_at >= DATEADD('day', -7, CURRENT_DATE())
);
```
